# 코호트 기준

## (1) 시간 기반 코호트

기준 : 가입 날짜

In [ ]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

In [ ]:
# 가상 데이터 생성
np.random.seed(42)

data = {
    'customer_id' : range(1, 501),
    'signup_date' : random.sample(pd.date_range(start='2023-01-01', periods=500, freq='D').tolist(), 500),
    'total_payment' : np.random.randint(100, 500, 500)
}

df = pd.DataFrame(data)
df.head()

,customer_id,signup_date,total_payment
0,1,2023-02-12,202
1,2,2023-08-06,448
2,3,2023-07-23,370
3,4,2023-12-24,206
4,5,2024-01-22,171


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    500 non-null    int64         
 1   signup_date    500 non-null    datetime64[ns]
 2   total_payment  500 non-null    int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 11.8 KB


In [ ]:
# 코호트 정의 : 고객의 가입 일자에서 년월 추출하여 코호트
df['signup_based_cohort'] = df['signup_date'].dt.to_period('M')

In [ ]:
df.head()

,customer_id,signup_date,total_payment,signup_based_cohort
0,1,2023-02-12,202,2023-02
1,2,2023-08-06,448,2023-08
2,3,2023-07-23,370,2023-07
3,4,2023-12-24,206,2023-12
4,5,2024-01-22,171,2024-01


In [ ]:
df

,customer_id,signup_date,total_payment,signup_based_cohort
0,1,2023-02-12,202,2023-02
1,2,2023-08-06,448,2023-08
2,3,2023-07-23,370,2023-07
3,4,2023-12-24,206,2023-12
4,5,2024-01-22,171,2024-01
...,...,...,...,...
495,496,2023-02-04,217,2023-02
496,497,2023-03-09,147,2023-03
497,498,2023-12-21,188,2023-12
498,499,2023-06-15,336,2023-06


## (2) 행동 기반 코호트

기준 : 첫 구매 상품

In [ ]:
# 가상 데이터 생성
np.random.seed(42)

data = {
    'customer_id' : np.random.randint(1, 101, 500),
    'order_date' : random.sample(pd.date_range(start='2023-01-01', periods=500, freq='D').tolist(), 500),
    'product_id' : [np.random.choice([101, 102, 103, 104, 105]) for i in range(500)],
    'total_payment' : np.random.randint(100, 500, 500)
}

df = pd.DataFrame(data)
df.head()

,customer_id,order_date,product_id,total_payment
0,52,2023-05-15,104,364
1,93,2023-01-12,102,498
2,15,2023-12-15,103,137
3,72,2024-01-22,105,193
4,61,2023-06-11,101,194


In [ ]:
# 고객별 첫 구매 날짜
first_purchase = df.groupby('customer_id')['order_date'].min().reset_index()
first_purchase.columns = ['customer_id', 'first_purchase_date']
first_purchase

,customer_id,first_purchase_date
0,1,2023-01-05
1,2,2023-03-01
2,3,2023-01-21
3,4,2023-01-09
4,5,2023-07-09
...,...,...
94,96,2023-08-31
95,97,2023-02-27
96,98,2023-09-12
97,99,2023-04-13


In [ ]:
# 고객 ID와 첫 구매 날짜를 사용하여 첫 구매 상품 찾기
first_product = pd.merge(first_purchase,
                        df,
                        left_on = ['customer_id', 'first_purchase_date'],
                        right_on = ['customer_id', 'order_date'],
                        how = 'left')
first_product.head()

,customer_id,first_purchase_date,order_date,product_id,total_payment
0,1,2023-01-05,2023-01-05,103,110
1,2,2023-03-01,2023-03-01,104,310
2,3,2023-01-21,2023-01-21,102,417
3,4,2023-01-09,2023-01-09,102,175
4,5,2023-07-09,2023-07-09,104,401


In [ ]:
first_product = first_product[['customer_id', 'product_id']]
first_product.columns = ['customer_id', 'first_product_cohort']

In [ ]:
first_product

,customer_id,first_product_cohort
0,1,103
1,2,104
2,3,102
3,4,102
4,5,104
...,...,...
94,96,102
95,97,104
96,98,104
97,99,102


In [ ]:
# 원본 데이터에 첫 구매 상품 정보 추가
df = pd.merge(df,
              first_product,
              on = 'customer_id',
              how = 'left')

In [ ]:
df.head()

,customer_id,order_date,product_id,total_payment,first_product_cohort
0,52,2023-05-15,104,364,104
1,93,2023-01-12,102,498,102
2,15,2023-12-15,103,137,103
3,72,2024-01-22,105,193,105
4,61,2023-06-11,101,194,105


## (3) 인구 통계학적 코호트

기준 : 성별 X 연령대

In [ ]:
# 가상 데이터 생성
np.random.seed(42)

data = {
    'customer_id' : range(1, 101),
    'age' : np.random.randint(18, 70, 100),
    'gender' : np.random.choice(['Male', 'Female'], 100),
    'total_payment' : np.random.randint(100, 1000, 100)
}

df = pd.DataFrame(data)
df.head()

,customer_id,age,gender,total_payment
0,1,56,Male,439
1,2,69,Male,897
2,3,46,Male,430
3,4,32,Male,739
4,5,60,Male,605


In [ ]:
# 연령대 컬럼 추가
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 20, 30, 40, 50, 60, 70],
                          labels=['10s', '20s', '30s', '40s', '50s', '60s'])

In [ ]:
df['cohort'] = df.apply(lambda x: f"{x['age_group']}_{x['gender']}", axis=1)

In [ ]:
df.head()

,customer_id,age,gender,total_payment,age_group,cohort
0,1,56,Male,439,50s,50s_Male
1,2,69,Male,897,60s,60s_Male
2,3,46,Male,430,40s,40s_Male
3,4,32,Male,739,30s,30s_Male
4,5,60,Male,605,50s,50s_Male


# 코호트 분석

코호트 정의 : 첫 구매(년월)

In [ ]:
# 가상 데이터 생성
np.random.seed(0)

n_customers = 500
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = (end_date - start_date).days

data = {
    'customer_id' : np.random.choice(range(1, n_customers+1), size=1000),
    'purchase_date' : [start_date + timedelta(days=np.random.randint(date_range)) for _ in range(1000)],
    'purchase_amount' : np.random.uniform(100, 1000, size=1000)
}

df = pd.DataFrame(data)
df.head()

,customer_id,purchase_date,purchase_amount
0,173,2024-06-11,377.675164
1,48,2024-02-20,947.966247
2,118,2024-08-30,899.438536
3,193,2024-12-24,874.279611
4,324,2024-01-21,687.699785


In [ ]:
# 코호트 정의 위해 매달 1일로 지정
df['purchase_month'] = df['purchase_date'].apply(lambda x: x.replace(day=1))

In [ ]:
df.head()

,customer_id,purchase_date,purchase_amount,purchase_month
0,173,2024-06-11,377.675164,2024-06-01
1,48,2024-02-20,947.966247,2024-02-01
2,118,2024-08-30,899.438536,2024-08-01
3,193,2024-12-24,874.279611,2024-12-01
4,324,2024-01-21,687.699785,2024-01-01


In [ ]:
# 고객별 첫 구매 날짜
df['first_purchase_month'] = df.groupby('customer_id')['purchase_date'].transform('min').apply(lambda x: x.replace(day=1))

In [ ]:
df.head()

,customer_id,purchase_date,purchase_amount,purchase_month,first_purchase_month
0,173,2024-06-11,377.675164,2024-06-01,2024-02-01
1,48,2024-02-20,947.966247,2024-02-01,2024-02-01
2,118,2024-08-30,899.438536,2024-08-01,2024-08-01
3,193,2024-12-24,874.279611,2024-12-01,2024-12-01
4,324,2024-01-21,687.699785,2024-01-01,2024-01-01


In [ ]:
# 첫 구매 이후 몇 개월이 지났는지
df['months_since_first_purchase'] = ((df['purchase_month'].dt.year - df['first_purchase_month'].dt.year) * 12
                                    + (df['purchase_month'].dt.month - df['first_purchase_month'].dt.month))

In [ ]:
df.head()

,customer_id,purchase_date,purchase_amount,purchase_month,first_purchase_month,months_since_first_purchase
0,173,2024-06-11,377.675164,2024-06-01,2024-02-01,4
1,48,2024-02-20,947.966247,2024-02-01,2024-02-01,0
2,118,2024-08-30,899.438536,2024-08-01,2024-08-01,0
3,193,2024-12-24,874.279611,2024-12-01,2024-12-01,0
4,324,2024-01-21,687.699785,2024-01-01,2024-01-01,0


In [ ]:
cohort_data = (df.groupby(['first_purchase_month', 'months_since_first_purchase'])
                  .agg(n_customers=('customer_id', 'nunique'),
                      total_purchase=('purchase_amount', 'sum'))
                  .reset_index())

In [ ]:
cohort_data.head()

,first_purchase_month,months_since_first_purchase,n_customers,total_purchase
0,2024-01-01,0,69,41118.192511
1,2024-01-01,1,9,5061.300569
2,2024-01-01,2,9,5348.686603
3,2024-01-01,3,14,8494.325584
4,2024-01-01,4,13,6973.269402


In [ ]:
# 코호트 피벗 테이블
cohort_pivot = cohort_data.pivot_table(index='first_purchase_month',
                                      columns='months_since_first_purchase',
                                      values='n_customers')
cohort_pivot

months_since_first_purchase,0,1,2,3,4,5,6,7,8,9,10,11
first_purchase_month,,,,,,,,,,,,
2024-01-01,69.0,9.0,9.0,14.0,13.0,13.0,10.0,11.0,12.0,10.0,10.0,8.0
2024-02-01,60.0,7.0,10.0,11.0,8.0,7.0,8.0,16.0,9.0,8.0,5.0,NaN
2024-03-01,69.0,11.0,14.0,2.0,8.0,11.0,10.0,9.0,9.0,15.0,NaN,NaN
2024-04-01,63.0,13.0,6.0,9.0,12.0,16.0,11.0,13.0,8.0,NaN,NaN,NaN
2024-05-01,30.0,3.0,3.0,4.0,4.0,7.0,4.0,2.0,NaN,NaN,NaN,NaN
2024-06-01,36.0,6.0,5.0,5.0,3.0,4.0,3.0,NaN,NaN,NaN,NaN,NaN
2024-07-01,29.0,7.0,3.0,3.0,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
2024-08-01,18.0,3.0,1.0,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-01,14.0,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 코호트별 사이즈
cohort_size = cohort_pivot.iloc[:, 0]
cohort_size

,0
first_purchase_month,
2024-01-01,69.0
2024-02-01,60.0
2024-03-01,69.0
2024-04-01,63.0
2024-05-01,30.0
2024-06-01,36.0
2024-07-01,29.0
2024-08-01,18.0
2024-09-01,14.0


시간이 지남에 따라 첫 구매를 하는 신규 사용자 유입이 감소하고 있으므로, 초기 유입과 첫 구매를 높이는 전략이 필요하다.

In [ ]:
# 유지율 계산
retention_matrix = np.round(cohort_pivot.divide(cohort_size, axis=0) * 100, 2)
retention_matrix

months_since_first_purchase,0,1,2,3,4,5,6,7,8,9,10,11
first_purchase_month,,,,,,,,,,,,
2024-01-01,100.0,13.04,13.04,20.29,18.84,18.84,14.49,15.94,17.39,14.49,14.49,11.59
2024-02-01,100.0,11.67,16.67,18.33,13.33,11.67,13.33,26.67,15.00,13.33,8.33,NaN
2024-03-01,100.0,15.94,20.29,2.90,11.59,15.94,14.49,13.04,13.04,21.74,NaN,NaN
2024-04-01,100.0,20.63,9.52,14.29,19.05,25.40,17.46,20.63,12.70,NaN,NaN,NaN
2024-05-01,100.0,10.00,10.00,13.33,13.33,23.33,13.33,6.67,NaN,NaN,NaN,NaN
2024-06-01,100.0,16.67,13.89,13.89,8.33,11.11,8.33,NaN,NaN,NaN,NaN,NaN
2024-07-01,100.0,24.14,10.34,10.34,6.90,20.69,NaN,NaN,NaN,NaN,NaN,NaN
2024-08-01,100.0,16.67,5.56,NaN,27.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-01,100.0,7.14,7.14,14.29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- 대부분의 코호트에서 시간이 지남에 따라 유지율이 감소하는 패턴을 보인다. 이는 시간이 지남에 따라 많은 사용자가 재구매를 하지 않게 된다는 점을 시사한다.
- 몇몇 코호트에서 중간에 유지율이 일시적으로 상승하는 패턴을 보이는데, 해당 시기에 특정 프로모션, 이벤트, 또는 특정 제품의 인기가 높아졌을 가능성을 시사한다.
- 몇몇 초기 코호트(예 : 2024년 1월 첫 구매 코호트)에서는 비교적 안정적인 유지율을 보인다. 장기적으로 높은 충성도를 보이는 사용자들이 주로 어떤 제품이나 서비스를 이용하는지 분석해볼 수 있다.